# 🚀 Servidor OpenCode GPU (Qwen 2.5 Coder) - Google AI Pro

### ⚡ Instrucciones:
1. Verificá que arriba a la derecha esté conectada la **GPU T4** (si dice *Conectar*, dale clic).
2. Hacé clic en el botón **Play** de la celda de abajo.
3. En ~1 minuto verás el cartel verde con tu enlace listo para copiar.

In [ ]:
import os, time, re, subprocess

print("==================================================================")
print("1/4 📦 Instalando Ollama...")
print("==================================================================")
!curl -fsSL https://ollama.ai/install.sh | sh > /dev/null 2>&1

print("\n==================================================================")
print("2/4 🌐 Descargando Cloudflare Tunnel...")
print("==================================================================")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

print("\n==================================================================")
print("3/4 ⚡ Iniciando motor Ollama y descargando Qwen 2.5 Coder...")
print("==================================================================")
!pkill -f "ollama serve" || true
!pkill -f "cloudflared" || true
time.sleep(2)

os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
os.environ["OLLAMA_ORIGINS"] = "*"
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)

# Descargar Qwen 2.5 Coder
!ollama pull qwen2.5-coder:7b

print("\n==================================================================")
print("4/4 🚀 Levantando túnel público de Cloudflare...")
print("==================================================================")
!rm -f tunnel.log
subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:11434"], stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)

public_url = None
for _ in range(40):
    time.sleep(1)
    if os.path.exists("tunnel.log"):
        with open("tunnel.log", "r") as f:
            content = f.read()
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
            if match:
                public_url = match.group(0)
                break

if public_url:
    print("\n" + "="*65)
    print("🟢 ¡TU SERVIDOR OPENCODE ESTÁ ACTIVO EN LA GPU DE GOOGLE!")
    print("="*65)
    print(f"\n👉 URL DEL TÚNEL PARA COPIAR:")
    print(f"   {public_url}\n")
    print(f"👉 MODELO: qwen2.5-coder:7b (Nvidia T4 GPU)")
    print("="*65)
    print("\n📋 Copiá la URL de arriba y pegala en el chat.")
    print("="*65 + "\n")
    
    try:
        while True:
            time.sleep(60)
    except KeyboardInterrupt:
        print("Túnel detenido.")
else:
    print("⚠️ No se encontró la URL en tunnel.log. Contenido:")
    if os.path.exists("tunnel.log"):
        with open("tunnel.log", "r") as f:
            print(f.read())
